## Activation Distribution Plot — Core Idea

An activation distribution plot visualizes the distribution of neuron outputs (activations) at different layers of a neural network.

### ACTIVATION DISTRIBUTION — MASTER CODE (ALL VARIATIONS)

We’ll:

- Build a small neural network

- Extract layer activations

- Plot distributions in multiple ways

In [ ]:
! pip install tensorflow

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.datasets import make_classification

sns.set(style="whitegrid")
np.random.seed(42)
tf.random.set_seed(42)


ModuleNotFoundError: No module named 'tensorflow'

In [2]:
# Synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_classes=2,
    random_state=42
)


NameError: name 'make_classification' is not defined

### Define Neural Network (ReLU Example)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(20,), name="Input"),
    layers.Dense(64, activation="relu", name="Dense_1"),
    layers.Dense(32, activation="relu", name="Dense_2"),
    layers.Dense(1, activation="sigmoid", name="Output")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

model.fit(X, y, epochs=10, batch_size=32, verbose=0)


### Build Activation Model (Intermediate Outputs)

In [ ]:
activation_model = models.Model(
    inputs=model.input,
    outputs=[layer.output for layer in model.layers]
)

activations = activation_model.predict(X)


### Basic Activation Distribution (Histogram)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(
    activations[1].flatten(),
    bins=50,
    color="steelblue",
    alpha=0.7
)
plt.title("Activation Distribution – Dense_1 (ReLU)")
plt.xlabel("Activation value")
plt.ylabel("Frequency")
plt.show()


### KDE / Smooth Density Plot (Recommended)

In [ ]:
plt.figure(figsize=(6, 4))
sns.kdeplot(
    activations[1].flatten(),
    fill=True
)
plt.title("Activation Density – Dense_1")
plt.xlabel("Activation value")
plt.show()


### Activation Distributions Across Layers (Overlay)

In [ ]:
plt.figure(figsize=(7, 5))

for i, layer_act in enumerate(activations[1:-1], start=1):
    sns.kdeplot(
        layer_act.flatten(),
        label=f"Dense_{i}"
    )

plt.title("Activation Distribution Across Layers")
plt.xlabel("Activation value")
plt.legend()
plt.show()


### Boxplot View (Layer-wise Summary)

In [ ]:
activation_data = [
    act.flatten() for act in activations[1:-1]
]

plt.figure(figsize=(6, 4))
plt.boxplot(activation_data, labels=["Dense_1", "Dense_2"])
plt.title("Activation Distribution (Boxplot)")
plt.ylabel("Activation value")
plt.show()


### Detect Dead ReLU Neurons

In [ ]:
dead_ratio = np.mean(activations[1] == 0)

print(f"Dead neuron ratio (Dense_1): {dead_ratio:.3f}")


### Compare Activations: ReLU vs Tanh

In [ ]:
def build_model(activation):
    m = models.Sequential([
        layers.Input(shape=(20,)),
        layers.Dense(64, activation=activation),
        layers.Dense(32, activation=activation),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    m.fit(X, y, epochs=5, verbose=0)
    return m

relu_model = build_model("relu")
tanh_model = build_model("tanh")

relu_act = models.Model(
    relu_model.input, relu_model.layers[1].output
).predict(X)

tanh_act = models.Model(
    tanh_model.input, tanh_model.layers[1].output
).predict(X)

plt.figure(figsize=(7, 4))
sns.kdeplot(relu_act.flatten(), label="ReLU")
sns.kdeplot(tanh_act.flatten(), label="Tanh")
plt.title("Activation Distribution: ReLU vs Tanh")
plt.xlabel("Activation value")
plt.legend()
plt.show()
